<a href="https://colab.research.google.com/github/tonHS/Canada_City_Stats/blob/main/Canadian_City_Stats_Hand_pulled.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ============================================================================
# Install Dependencies and Packages
# ============================================================================
!pip install stats-can openpyxl

import pandas as pd
import requests
import zipfile
from io import BytesIO
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

In [9]:
# ============================================================================
# SETUP: Create directory
# ============================================================================
print(">>> ENTERING SETUP")

data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
outputs_dir = Path('outputs')
outputs_dir.mkdir(exist_ok=True)

>>> ENTERING SETUP


In [10]:
# ============================================================================
# FETCH: CMA Employment Data
# ============================================================================
print(">>> ENTERING: CMA Employment Data Fetch")
print("=" * 80)
print("FETCHING CMA EMPLOYMENT DATA FROM STATISTICS CANADA")
print("=" * 80)

TABLE_ID = "14100459"
download_url = f"https://www150.statcan.gc.ca/n1/tbl/csv/{TABLE_ID}-eng.zip"

print(f"\n📥 Downloading data from Statistics Canada (Table {TABLE_ID})...")
response = requests.get(download_url, timeout=30)
response.raise_for_status()

with zipfile.ZipFile(BytesIO(response.content)) as zip_file:
    csv_files = [f for f in zip_file.namelist() if f.endswith('.csv')]
    csv_filename = csv_files[0]
    with zip_file.open(csv_filename) as csv_file:
        df_jobs = pd.read_csv(csv_file)

print(f"✓ Data loaded: {len(df_jobs):,} rows, {len(df_jobs.columns)} columns")

# Save raw data
raw_data_path = data_dir / 'cma_employment_raw.csv'
df_jobs.to_csv(raw_data_path, index=False)
print(f"✓ Raw data saved to: {raw_data_path}")
print(">>> CMA Employment Data Fetch COMPLETE")

df_jobs.head()



>>> ENTERING: CMA Employment Data Fetch
FETCHING CMA EMPLOYMENT DATA FROM STATISTICS CANADA

📥 Downloading data from Statistics Canada (Table 14100459)...


/tmp/ipykernel_6335/1650586262.py:20: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df_jobs = pd.read_csv(csv_file)


✓ Data loaded: 313,524 rows, 17 columns
✓ Raw data saved to: data/cma_employment_raw.csv
>>> CMA Employment Data Fetch COMPLETE


,REF_DATE,GEO,DGUID,Labour force characteristics,Statistics,Data type,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2011-01,Canada,2021A000011124,Population,Estimate,Seasonally adjusted,Persons in thousands,428,units,0,v1643277902,1.1.1.1,27736.9,NaN,NaN,NaN,1
1,2011-01,Canada,2021A000011124,Population,Estimate,Unadjusted,Persons in thousands,428,units,0,v1643277903,1.1.1.2,27736.9,NaN,NaN,NaN,1
2,2011-01,Canada,2021A000011124,Labour force,Estimate,Seasonally adjusted,Persons in thousands,428,units,0,v1643277910,1.2.1.1,18588.0,NaN,NaN,NaN,1
3,2011-01,Canada,2021A000011124,Labour force,Estimate,Unadjusted,Persons in thousands,428,units,0,v1643277911,1.2.1.2,18395.8,NaN,NaN,NaN,1
4,2011-01,Canada,2021A000011124,Employment,Estimate,Seasonally adjusted,Persons in thousands,428,units,0,v1643277918,1.3.1.1,17131.3,NaN,NaN,NaN,1


In [ ]:
# ============================================================================
# FETCH: City Population Data
# ============================================================================

print(">>> ENTERING: City Population Data Fetch")
print("=" * 80)
print("FETCHING City Population Data from Statistics Canada")
print("=" * 80)

TABLE_ID = "17100148"
download_url = f"https://www150.statcan.gc.ca/n1/tbl/csv/{TABLE_ID}-eng.zip"

print(f"\n📥 Downloading data from Statistics Canada (Table {TABLE_ID})...")
response = requests.get(download_url, timeout=30)
response.raise_for_status()

with zipfile.ZipFile(BytesIO(response.content)) as zip_file:
    csv_files = [f for f in zip_file.namelist() if f.endswith('.csv')]
    csv_filename = csv_files[0]
    with zip_file.open(csv_filename) as csv_file:
        df_ppn = pd.read_csv(csv_file)

print(f"✓ Data loaded: {len(df):,} rows, {len(df.columns)} columns")

# Save raw data
raw_data_path = data_dir / 'city_population_data.csv'
df_ppn.to_csv(raw_data_path, index=False)
print(f"✓ Raw data saved to: {raw_data_path}")
print(">>> City Population Data Fetch COMPLETE")

df_ppn.head()


>>> ENTERING: City Population Data Fetch
FETCHING City Population Data from Statistics Canada

📥 Downloading data from Statistics Canada (Table 17100148)...


In [ ]:
# ============================================================================
# PROCESS: CMA Employment Data
# ============================================================================

# Filter for the employment numbers and unemployment rates of the 13 dashboard CMAs (census agglomerations not included in this data)

dashboard_cmas = [
    'Toronto, Ontario',
    'Montréal, Quebec',
    'Vancouver, British Columbia',
    'Calgary, Alberta',
    'Edmonton, Alberta',
    'Ottawa-Gatineau, Ontario/Quebec',
    'Winnipeg, Manitoba',
    'Quebec, Quebec',
    'Hamilton, Ontario',
    'Halifax, Nova Scotia',
    'Saskatoon, Saskatchewan',
    'Fredericton, New Brunswick',
    "St. John's, Newfoundland and Labrador"
]


# Filter df_jobs for the dashboard CMAs, employment and unemployment rate
df_jobs_filtered = df_jobs[
    (df_jobs['GEO'].isin(dashboard_cmas)) &
    (df_jobs['Labour force characteristics'].isin(['Employment', 'Unemployment rate'])) &
    (df_jobs['Statistics'] == 'Estimate') &
    (df_jobs['Data type'].isin(['Seasonally adjusted'])) &
    (df_jobs['REF_DATE'] == df_jobs['REF_DATE'].max()) # Corrected filter for most recent month
].copy()

# Select relevant columns
df_jobs_filtered = df_jobs_filtered[[
    'REF_DATE',
    'GEO',
    'Labour force characteristics',
    'VALUE'
]].copy()

# Rename columns for clarity
df_jobs_filtered.rename(columns={
    'REF_DATE': 'Date',
    'GEO': 'City',
    'Labour force characteristics': 'Characteristic',
    'VALUE': 'Value'
}, inplace=True)

print(f"Filtered employment data for {len(dashboard_cmas)} dashboard CMAs for the most recent month.")
print(f"Data loaded: {len(df_jobs_filtered):,} rows, {len(df_jobs_filtered.columns)} columns.")

df_jobs_filtered.head(20)

In [ ]:
import pandas as pd

# ============================================================================
# PROCESS: City Population Data
# ============================================================================

# Filter for the populations of the 17 dashboard cities, both CMAs and census agglomerations


dashboard_citypop = [
    'Toronto (CMA), Ontario',
    'Montréal (CMA), Quebec',
    'Vancouver (CMA), British Columbia',
    'Calgary (CMA), Alberta',
    'Edmonton (CMA), Alberta',
    'Ottawa - Gatineau (Ontario part), Ontario/Quebec', # Corrected name for Ottawa-Gatineau
    'Winnipeg (CMA), Manitoba',
    'Québec, Quebec', # Corrected name for Quebec
    'Hamilton (CMA), Ontario',
    'Halifax (CMA), Nova Scotia',
    'Saskatoon (CMA), Saskatchewan',
    'Fredericton (CMA), New Brunswick',
    "St. John's (CMA), Newfoundland and Labrador",
    'Charlottetown (CA), Prince Edward Island',
    'Yellowknife (CA), Northwest Territories',
    'Whitehorse (CA), Yukon',
    #Iqualuit is sole-sourced from a census dataset

]

# Convert REF_DATE in df_ppn to datetime objects to ensure proper comparison
df_ppn['REF_DATE'] = pd.to_datetime(df_ppn['REF_DATE'], format='%Y')

# Get the maximum year from the population data
max_ppn_year = df_ppn['REF_DATE'].max()


# Filter df_ppn for the dashboard cities minus Iqualuit
df_ppn_filtered = df_ppn[
    (df_ppn['GEO'].isin(dashboard_citypop)) &
    (df_ppn['REF_DATE'] == max_ppn_year) & # Filter by the max year from df_ppn
    (df_ppn['Gender'] == 'Total - gender') &
    (df_ppn['Age group'] == 'All ages')

]

# Select relevant columns
df_ppn_filtered = df_ppn_filtered[[
    'REF_DATE',
    'GEO',
    'VALUE'
]].copy()


# Rename columns for clarity
df_ppn_filtered.rename(columns={
    'REF_DATE': 'Date',
    'GEO': 'City',
    'VALUE': 'Population'
}, inplace=True)

print(f"Filtered population data for {len(dashboard_citypop)} dashboard cities for the most recent year.")
print(f"Data loaded: {len(df_ppn_filtered):,} rows, {len(df_ppn_filtered.columns)} columns.")

df_ppn_filtered.head(20)